# Used Car Dealership Market Analysis

This notebook performs a comprehensive exploratory data analysis (EDA) across three markets: Los Angeles, Dallas–Fort Worth, and San Juan.
It includes interactive visualizations, buyer demographic analysis, and inventory recommendations.

In [1]:
# Import required libraries
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display, clear_output
import ipywidgets as widgets

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

## 1. Load Dataset
Load the dataset

In [2]:
df = pd.read_csv('car_sales_data.csv', parse_dates=['sale_date'])
df.head()

## 2. Data Cleaning
Check for missing values

In [3]:
df.isnull().sum()

### Fill missing values
Fill missing `buyer_annual_income` by median per market and `payment_to_income_ratio` by median per purchase type.

In [4]:
df['buyer_annual_income'] = df.groupby('market')['buyer_annual_income'].transform(lambda x: x.fillna(x.median()))
df['payment_to_income_ratio'] = df.groupby('purchase_type')['payment_to_income_ratio'].transform(lambda x: x.fillna(x.median()))

## 3. General Overview
Overview of total sales, vehicle tiers, and body styles across all markets.

In [5]:
fig = px.histogram(df, x='market', title='Total Sales by Market', color='market')
fig.show()

### Vehicle Tier Distribution
Proportion of `mass` vs `luxury` cars sold in each market.

In [6]:
fig = px.histogram(df, x='vehicle_tier', color='market', barmode='group', title='Vehicle Tier Distribution by Market')
fig.show()

### Body Style Distribution
Distribution of vehicle body styles sold in each market.

In [7]:
fig = px.histogram(df, x='body_style', color='market', barmode='group', title='Body Style Distribution by Market')
fig.show()

## 4. Interactive Market, Tier, Body Style Filters
Explore sales, top models, and price vs mileage interactively using filters.

In [8]:
# Widgets
market_widget = widgets.Dropdown(options=['All'] + list(df['market'].unique()), value='All', description='Market:')
tier_widget = widgets.Dropdown(options=['All'] + list(df['vehicle_tier'].unique()), value='All', description='Vehicle Tier:')
body_widget = widgets.Dropdown(options=['All'] + list(df['body_style'].unique()), value='All', description='Body Style:')

def update_plots(market, tier, body):
    clear_output(wait=True)
    display(market_widget, tier_widget, body_widget)
    df_filtered = df.copy()
    if market != 'All': df_filtered = df_filtered[df_filtered['market']==market]
    if tier != 'All': df_filtered = df_filtered[df_filtered['vehicle_tier']==tier]
    if body != 'All': df_filtered = df_filtered[df_filtered['body_style']==body]
    # Total Sales
    fig1 = px.histogram(df_filtered, x='market', color='market', title='Total Sales by Market')
    fig1.show()
    # Vehicle Tier
    fig2 = px.histogram(df_filtered, x='vehicle_tier', color='market', barmode='group', title='Vehicle Tier Distribution')
    fig2.show()
    # Body Style
    fig3 = px.histogram(df_filtered, x='body_style', color='market', barmode='group', title='Body Style Distribution')
    fig3.show()
    # Top Models
    if not df_filtered.empty:
        top_models = df_filtered.groupby(['make','model']).agg(sales_count=('sale_id','count'), avg_price=('price','mean'), avg_mileage=('mileage','mean')).reset_index().sort_values('sales_count', ascending=False).head(10)
        fig4 = px.bar(top_models, x='model', y='sales_count', color='make', hover_data=['avg_price','avg_mileage'], title='Top 10 Models')
        fig4.show()
    # Price vs Mileage Scatter
    fig5 = px.scatter(df_filtered, x='mileage', y='price', color='vehicle_tier', hover_data=['make','model'], opacity=0.5, title='Price vs Mileage')
    fig5.show()

widgets.interact(update_plots, market=market_widget, tier=tier_widget, body=body_widget)

## 5. Buyer Demographics & Purchase Behavior
Analyze buyer age, income, and purchase type across markets.

In [9]:
# Buyer Age vs Purchase Type
fig = px.box(df, x='purchase_type', y='buyer_age', color='market', title='Buyer Age Distribution by Purchase Type & Market')
fig.show()

# Buyer Income vs Purchase Type
fig = px.box(df, x='purchase_type', y='buyer_annual_income', color='market', title='Income Distribution by Purchase Type & Market')
fig.show()

# Purchase Type by Age Group
age_purchase = df.groupby(['market','buyer_age_group','purchase_type']).size().reset_index(name='count')
fig = px.bar(age_purchase, x='buyer_age_group', y='count', color='purchase_type', facet_col='market', barmode='stack', title='Purchase Type by Buyer Age Group & Market')
fig.show()

## 6. Market Inventory Recommendations
Recommend top models to stock per market based on historical sales.

In [27]:
import plotly.express as px

# Define the analysis period
start_date = df['sale_date'].min().strftime('%Y-%m-%d')
end_date = df['sale_date'].max().strftime('%Y-%m-%d')

# Aggregate units sold and revenue per model per market
inventory_df = (
    df.groupby(['market', 'model'], as_index=False)
      .agg(
          units_sold=('sale_id', 'count'),
          revenue_generated=('price', 'sum')
      )
)

# Compute percentage of total market units (market share)
inventory_df['units_pct'] = inventory_df.groupby('market')['units_sold'].transform(lambda x: 100 * x / x.sum())
inventory_df['revenue_pct'] = inventory_df.groupby('market')['revenue_generated'].transform(lambda x: 100 * x / x.sum())

# Plot top 10 models per market using % of market units as bar length
markets = df['market'].unique()

for market in markets:
    df_market = inventory_df[inventory_df['market'] == market].sort_values('units_sold', ascending=False).head(10)
    
    fig = px.bar(
        df_market,
        x='units_pct',  # Bar length = % of market units
        y='model',
        orientation='h',
        text=df_market['units_sold'],  # show actual units sold on the bar
        color='revenue_generated',
        color_continuous_scale='Viridis',
        title=f'Top 10 Models in {market} by Market Share & Revenue\nPeriod: {start_date} to {end_date}',
        labels={
            'units_pct':'% of Market Units',
            'revenue_generated':'Revenue ($)',
            'model':'Vehicle Model'
        }
    )
    
    # Add detailed hover info
    fig.update_traces(
        hovertemplate=(
            '<b>%{y}</b><br>'
            'Units Sold: %{text} (%{x:.2f}%)<br>'
            'Revenue: $%{customdata[0]:,.0f} (%{customdata[1]:.2f}%)'
        ),
        customdata=df_market[['revenue_generated','revenue_pct']]
    )
    
    fig.update_layout(yaxis={'categoryorder':'total ascending'})
    fig.show()


## 7. Summary & Recommendations
1. **Los Angeles**: Focus on luxury vehicles (BMW, Mercedes-Benz, Tesla).
2. **Dallas–Fort Worth**: Mass-market trucks & SUVs perform best (F-150, Silverado, CR-V).
3. **San Juan**: Affordable sedans and SUVs for lower-income buyers.
4. Younger buyers (18-34) are more likely to lease; older buyers prefer finance or cash.